# Data Preparation

## Current Problem

Our data has an imbalance of 94:6 (no claim : claim)

We want to have a similar amount of labels or classes that would help us identify claim and no claim. Having the majority as claim would not give enough data to our model to how a "no claim" policy holder would look like. 

Hence, we employ data-level techniques such as:

- SMOTE (Synthetic Minority Oversampling Technique)
    - Creates synthetic samples by interpolating between existing minority samples and their neighbors

- Random Undersampling

    - Removes majority class samples randomly

- Tomek Links
    - Remove pairs of examples from opposite classes that are nearest neighbors
    



In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, f1_score,
    roc_auc_score, precision_recall_curve, average_precision_score,
    roc_curve, precision_score, recall_score
)
import warnings
warnings.filterwarnings('ignore')

from imblearn.over_sampling import SMOTE, ADASYN, BorderlineSMOTE
from imblearn.under_sampling import RandomUnderSampler, TomekLinks
from imblearn.combine import SMOTETomek, SMOTEENN
from imblearn.pipeline import Pipeline as ImbPipeline
import kagglehub
import os


In [2]:
path = kagglehub.dataset_download("litvinenko630/insurance-claims")

# List all files in the downloaded directory
files = os.listdir(path)
file_path = os.path.join(path, "Insurance claims data.csv")
df = pd.read_csv(file_path)

In [4]:
from helpers.preprocess import *

df_processed = preprocess_data(df)
print(f"Processed shape: {df_processed.shape}")
print(f"\nFeature types after processing:")
print(df_processed.dtypes.value_counts())

NameError: name 'pd' is not defined

## SMOTE

1. For each minority sample, find its k nearest neighbors (default k=5)
2. Randomly select one of those neighbors
3. Create a new synthetic sample along the line connecting them

Formula: `new_sample = original + random(0,1) × (neighbor - original)` 

This creates realistic variations that help the model generalize better.

### Also has some Variants
#### BorderlineSMOTE
Only generates synthetic samples for minority instances that are **near the decision boundary** (close to majority samples). This focuses on the hard-to-classify cases.

#### ADASYN (Adaptive Synthetic Sampling)
Generates **more synthetic samples** for minority instances that are harder to classify (surrounded by majority samples). It's adaptive to the local density.


In [11]:
# Compare SMOTE variants
smote_variants = {
    'Standard SMOTE': SMOTE(random_state=42),
    'BorderlineSMOTE': BorderlineSMOTE(random_state=42),
    'ADASYN': ADASYN(random_state=42)
}

print("Comparison of SMOTE Variants:")
print("=" * 50)

for name, sampler in smote_variants.items():
    try:
        X_res, y_res = sampler.fit_resample(X_train_scaled, y_train)
        print(f"\n{name}:")
        print(f"  Total samples: {len(y_res)}")
        print(f"  Minority samples: {sum(y_res == 1)} (was {sum(y_train == 1)})")
        print(f"  New synthetic: {sum(y_res == 1) - sum(y_train == 1)}")
    except Exception as e:
        print(f"\n{name}: Error - {e}")

Comparison of SMOTE Variants:

Standard SMOTE: Error - name 'X_train_scaled' is not defined

BorderlineSMOTE: Error - name 'X_train_scaled' is not defined

ADASYN: Error - name 'X_train_scaled' is not defined


In [ ]:
# SMOTE + Tomek Links
smote_tomek = SMOTETomek(random_state=42)
X_train_st, y_train_st = smote_tomek.fit_resample(X_train_scaled, y_train)

print("SMOTE + Tomek Links Results:")
print(f"  Original: {len(y_train)} samples")
print(f"  After SMOTE: ~{sum(y_train==0) + sum(y_train==0)} (balanced)")
print(f"  After Tomek cleanup: {len(y_train_st)} samples")
print(f"  \nClass distribution:")
print(f"  No Claim: {sum(y_train_st == 0)}")
print(f"  Claim: {sum(y_train_st == 1)}")

In [ ]:
plt.figure(figsize=(6, 4))
counts = df['claim_status'].value_counts().sort_index()
plt.barh(counts.index.astype(str), counts.values, color=['#44624a', '#c0cfb2'])
plt.xlabel('Count')
plt.ylabel('Claim Status')
plt.title('Distribution of Claim Status')

plt.tight_layout()
plt.show()